In [1]:
!pip install mlxtend --quiet

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import scipy
import sklearn
cwd = Path('.')

In [3]:
data_path = cwd / 'data' / '2025_LoL_esports_match_data_from_OraclesElixir_imputated.csv'
data = pd.read_csv(data_path,index_col=0)
data_len = int(len(data) * 10 / 12)

/tmp/ipykernel_32012/4154608675.py:2: DtypeWarning: Columns (0: league) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path,index_col=0)


In [4]:
data = data.drop(data[data["position"] == "team"].index) 

In [5]:
data

,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,200.0,224.0,-1.0,17.0,1.0,1.0,2.0,2.0,4.0,2.0
1,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,157.0,-2363.0,-1444.0,-18.0,0.0,1.0,2.0,1.0,7.0,0.0
2,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,241.0,-1552.0,-2465.0,-41.0,1.0,0.0,2.0,1.0,5.0,1.0
3,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,257.0,-2613.0,-1156.0,-6.0,1.0,1.0,2.0,6.0,2.0,0.0
4,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,20.0,-662.0,-734.0,18.0,0.0,2.0,2.0,0.0,8.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120629,LOLTMNT03_332179,complete,NaN,DCup,2025,NaN,0,2025-12-29 10:52:27,3,15.24,...,188.0,1763.0,3518.0,31.0,2.0,5.0,2.0,0.0,5.0,3.0
120630,LOLTMNT03_332179,complete,NaN,DCup,2025,NaN,0,2025-12-29 10:52:27,3,15.24,...,144.0,1750.0,2364.0,45.0,2.0,8.0,1.0,1.0,4.0,3.0
120631,LOLTMNT03_332179,complete,NaN,DCup,2025,NaN,0,2025-12-29 10:52:27,3,15.24,...,224.0,-125.0,-500.0,-32.0,8.0,5.0,2.0,4.0,4.0,1.0
120632,LOLTMNT03_332179,complete,NaN,DCup,2025,NaN,0,2025-12-29 10:52:27,3,15.24,...,221.0,-541.0,513.0,5.0,0.0,7.0,2.0,2.0,4.0,2.0


Picks / bans

In [6]:
from collections import defaultdict

h = defaultdict(int)
pick_freq = defaultdict(int)

champions = data["champion"].tolist()

total_games = len(data) // 10
ban_cols = [data[f"ban{i}"].tolist() for i in range(1, 6)]

for i in range(0, len(data), 10):
    blue_picks = [champions[i + j]     + "_picked" for j in range(5)]
    red_picks  = [champions[i + 5 + j] + "_picked" for j in range(5)]

    # Reusing the same names for bans to reuse the code
    # blue_picks = list({ban_cols[c][i]     + "_banned" for c in range(5) if ban_cols[c][i]     == ban_cols[c][i]})
    # red_picks  = list({ban_cols[c][i + 5] + "_banned" for c in range(5) if ban_cols[c][i + 5] == ban_cols[c][i + 5]})

    if len(blue_picks) != 5 or len(red_picks) != 5:
        continue

    for p in blue_picks + red_picks:
       pick_freq[p] += 1

    #In different teams
    # for p in blue_picks:
    #    for q in red_picks:
    #        h[(p, q)] += 1

    # # In same team
    for i in range(len(blue_picks)):
        for j in range(i + 1, len(blue_picks)):
            h[(blue_picks[i], blue_picks[j])] += 1
            h[(red_picks[i], red_picks[j])] += 1


for k in list(h.keys()):
    p_ab = h[k] / total_games
    p_a = pick_freq[k[0]] / total_games
    p_b = pick_freq[k[1]] / total_games
    h[k] = p_ab / (p_a * p_b) if p_ab >= 0.01 else 0 # With popularity norm
    # h[k] = p_ab if p_ab >= 0.02 else 0 # Without popularity norm

freq_df = pd.DataFrame(list(h.items()), columns=["itemset", "norm_support"])
freq_df = freq_df.sort_values(by="norm_support", ascending=False)
freq_df

,itemset,norm_support
1642,"(Lucian_picked, Nami_picked)",8.819710
563,"(Kalista_picked, Renata Glasc_picked)",6.816475
2382,"(Zeri_picked, Lulu_picked)",4.564173
544,"(Xayah_picked, Rakan_picked)",2.338288
4491,"(Lucian_picked, Braum_picked)",2.119759
...,...,...
3629,"(Twitch_picked, Elise_picked)",0.000000
3630,"(Poppy_picked, Seraphine_picked)",0.000000
3631,"(Viego_picked, Seraphine_picked)",0.000000
3632,"(Ahri_picked, Seraphine_picked)",0.000000


In [7]:
from utils.transform import champion_class_transform, champion_to_tags
champion_class_transform(data.copy()) # Just de bug print the tags
# Find max number of tags in one champion
max_tags = 0
for champion in champions:
    tags = champion_to_tags(champion)
    max_tags = max(max_tags, len(tags))
print(max_tags)

['Fighter', 'Assassin', 'Mage', 'Marksman', 'Support', 'Tank']
2


Legacy tags (classes)

In [8]:
from collections import defaultdict
from utils.transform import champion_to_tags

h = defaultdict(int)
pick_freq = defaultdict(int)

tags = ['Assassin', 'Marksman', 'Fighter', 'Support', 'Mage', 'Tank']
tags_idx = {t: i for i, t in enumerate(tags)}

champions = data["champion"].tolist()

total_games = len(data) // 10
ban_cols = [data[f"ban{i}"].tolist() for i in range(1, 6)]

for i in tqdm(range(0, len(data), 10)):
    blue_picks = [champions[i + j]     for j in range(5)]
    red_picks  = [champions[i + 5 + j] for j in range(5)]

    blue_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in blue_picks]))
    red_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in red_picks]))
    h[blue_tags] += 1
    h[red_tags] += 1

for k in list(h.keys()):
    p_ab = h[k] / total_games
    # p_a = pick_freq[k[0]] / total_games
    # p_b = pick_freq[k[1]] / total_games
    # h[k] = p_ab / (p_a * p_b) if p_ab >= 0.02 else 0 # With popularity norm
    h[k] = p_ab if p_ab >= 0.001 else 0 # Without popularity norm

to_find_example = (('Assassin', 'Fighter'), ('Fighter', 'Tank'), ('Mage', 'Marksman'), ('Mage', 'Support'), ('Support', 'Tank'))

for i in tqdm(range(0, len(data), 10)):
    blue_picks = [champions[i + j]     for j in range(5)]
    red_picks  = [champions[i + 5 + j] for j in range(5)]

    blue_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in blue_picks]))
    red_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in red_picks]))

    if blue_tags == to_find_example:
        print(blue_picks)
        break
    if red_tags == to_find_example:
        print(red_picks)
        break

freq_df = pd.DataFrame(list(h.items()), columns=["itemset", "norm_support"])
freq_df = freq_df.sort_values(by="norm_support", ascending=False)
freq_df

  0%|          | 0/10053 [00:00<?, ?it/s]

  0%|          | 0/10053 [00:00<?, ?it/s]

['Renekton', 'Lee Sin', 'Taliyah', 'Corki', 'Rell']


,itemset,norm_support
34,"((Assassin, Fighter), (Fighter, Tank), (Mage, ...",0.050333
118,"((Fighter, Tank), (Fighter, Tank), (Mage, Mark...",0.033124
18,"((Assassin, Fighter), (Assassin, Mage), (Fight...",0.031036
27,"((Assassin, Fighter), (Fighter, Tank), (Mage,)...",0.028549
24,"((Fighter, Tank), (Fighter, Tank), (Mage,), (M...",0.023973
...,...,...
1088,"((Assassin, Marksman), (Fighter, Marksman), (F...",0.000000
1119,"((Fighter,), (Fighter, Tank), (Mage, Support),...",0.000000
1118,"((Assassin, Fighter), (Assassin, Fighter), (Fi...",0.000000
1117,"((Assassin,), (Assassin, Fighter), (Mage, Mark...",0.000000


Legacy tags means per team (i.e team archetypes)

In [9]:
from collections import defaultdict
from utils.transform import champion_to_tags

h = defaultdict(int)
pick_freq = defaultdict(int)

tags = ['Assassin', 'Marksman', 'Fighter', 'Support', 'Mage', 'Tank']
tags_idx = {t: i for i, t in enumerate(tags)}

champions = data["champion"].tolist()

total_games = len(data) // 10
ban_cols = [data[f"ban{i}"].tolist() for i in range(1, 6)]

for i in tqdm(range(0, len(data), 10)):
    blue_picks = [champions[i + j]     for j in range(5)]
    red_picks  = [champions[i + 5 + j] for j in range(5)]

    # blue_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in blue_picks]))
    # red_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in red_picks]))
    # h[blue_tags] += 1
    # h[red_tags] += 1
    blue_tags = [champion_to_tags(champion) for champion in blue_picks]
    red_tags = [champion_to_tags(champion) for champion in red_picks]

    blue_tags_freqs = [0] * len(tags)
    for tag in blue_tags:
        n_tags = len(tag)
        for t in tag:
            blue_tags_freqs[tags_idx[t]] += 2 // n_tags # Just use 2 so they are integers, maybe better hashing
            #blue_tags_freqs[tags_idx[t]] += 1
    h[tuple(blue_tags_freqs)] += 1

for k in list(h.keys()):
    p_ab = h[k] / total_games
    # p_a = pick_freq[k[0]] / total_games
    # p_b = pick_freq[k[1]] / total_games
    # h[k] = p_ab / (p_a * p_b) if p_ab >= 0.02 else 0 # With popularity norm
    h[k] = p_ab if p_ab >= 0.001 else 0 # Without popularity norm

# to_find_example = (('Assassin', 'Fighter'), ('Fighter', 'Tank'), ('Mage', 'Marksman'), ('Mage', 'Support'), ('Support', 'Tank'))

# for i in tqdm(range(0, len(data), 10)):
#     blue_picks = [champions[i + j]     for j in range(5)]
#     red_picks  = [champions[i + 5 + j] for j in range(5)]

#     blue_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in blue_picks]))
#     red_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in red_picks]))

#     if blue_tags == to_find_example:
#         print(blue_picks)
#     if red_tags == to_find_example:
#         print(red_picks)

print(tags)
freq_df = pd.DataFrame(list(h.items()), columns=["itemset", "norm_support"])
freq_df = freq_df.sort_values(by="norm_support", ascending=False)
freq_df

  0%|          | 0/10053 [00:00<?, ?it/s]

['Assassin', 'Marksman', 'Fighter', 'Support', 'Mage', 'Tank']


,itemset,norm_support
5,"(1, 1, 2, 2, 2, 2)",0.033721
48,"(1, 1, 2, 1, 3, 2)",0.025962
40,"(1, 2, 2, 1, 2, 2)",0.025664
9,"(2, 1, 2, 1, 2, 2)",0.022680
27,"(0, 1, 2, 2, 3, 2)",0.020392
...,...,...
278,"(1, 0, 3, 0, 2, 4)",0.000000
279,"(2, 3, 2, 0, 1, 2)",0.000000
280,"(1, 2, 1, 4, 1, 1)",0.000000
281,"(3, 1, 1, 1, 4, 0)",0.000000


New tags per team (i.e again team archetypes, but using the new classes, no need for mean cuz 1 per champion)

In [10]:
from collections import defaultdict
from utils.transform import champion_to_better_tags, champion_better_tags_json, _clean

h = defaultdict(int)
pick_freq = defaultdict(int)

better_tags = champion_better_tags_json.keys()
better_tags_idx = {t: i for i, t in enumerate(better_tags)}

champions = data["champion"].tolist()

total_games = len(data) // 10
ban_cols = [data[f"ban{i}"].tolist() for i in range(1, 6)]

for i in tqdm(range(0, len(data), 10)):
    blue_picks = [champions[i + j]     for j in range(5)]
    red_picks  = [champions[i + 5 + j] for j in range(5)]

    # blue_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in blue_picks]))
    # red_tags = tuple(sorted([tuple(sorted(champion_to_tags(champion))) for champion in red_picks]))
    # h[blue_tags] += 1
    # h[red_tags] += 1
    #blue_tags = tuple(sorted([champion_to_better_tags(champion) for champion in blue_picks]))
    #red_tags = tuple(sorted([champion_to_better_tags(champion) for champion in red_picks]))
    blue_tags = tuple([champion_to_better_tags(champion) for champion in blue_picks])
    red_tags = tuple([champion_to_better_tags(champion) for champion in red_picks])

    h[blue_tags] += 1
    h[red_tags] += 1

    #blue_tags_freqs = [0] * len(better_tags)
    #for tag in blue_tags:
    #    n_tags = len(tag)
    #    for t in tag:
    #        blue_tags_freqs[better_tags_idx[t]] += 1 // n_tags # Just use 2 so they are integers, maybe better hashing
    #        #blue_tags_freqs[better_tags_idx[t]] += 1

for k in list(h.keys()):
    p_ab = h[k] / total_games / 2
    # p_a = pick_freq[k[0]] / total_games
    # p_b = pick_freq[k[1]] / total_games
    # h[k] = p_ab / (p_a * p_b) if p_ab >= 0.02 else 0 # With popularity norm
    #h[k] = p_ab if p_ab >= 0.001 else 0 # Without popularity norm
    h[k] = p_ab

freq_df = pd.DataFrame(list(h.items()), columns=["itemset", "norm_support"])
freq_df = freq_df.sort_values(by="norm_support", ascending=False)
freq_df

  0%|          | 0/10053 [00:00<?, ?it/s]

,itemset,norm_support
19,"(Skirmisher, Diver, Burst, Marksman, Vanguard)",0.015070
27,"(Diver, Diver, Burst, Marksman, Vanguard)",0.012534
23,"(Diver, Diver, Battlemage, Marksman, Vanguard)",0.011986
81,"(Skirmisher, Diver, Battlemage, Marksman, Vang...",0.011091
247,"(Vanguard, Diver, Burst, Marksman, Vanguard)",0.009301
...,...,...
4040,"(Vanguard, Artillery, Assassin, Artillery, Van...",0.000050
4039,"(Skirmisher, Assassin, Assassin, Artillery, Ca...",0.000050
4037,"(Battlemage, Burst, Skirmisher, Marksman, Warden)",0.000050
4036,"(Diver, Assassin, Specialist, Catcher, Catcher)",0.000050


We expect the Specialist to be mostly Azir, as it is mostly the only champion played profesionally with the Specialist tag

In [11]:
#to_find_example = ('Diver', 'Diver', 'Burst', 'Marksman', 'Vanguard')
to_find_example = ('Skirmisher', 'Diver', 'Specialist', 'Marksman', 'Vanguard')

for i in tqdm(range(0, len(data), 10)):
    blue_picks = [champions[i + j]     for j in range(5)]
    red_picks  = [champions[i + 5 + j] for j in range(5)]

    blue_tags = tuple([champion_to_better_tags(champion) for champion in blue_picks])
    red_tags = tuple([champion_to_better_tags(champion) for champion in red_picks])

    if blue_tags == to_find_example:
        print(blue_picks[2])
    if red_tags == to_find_example:
        print(red_picks[2])

  0%|          | 0/10053 [00:00<?, ?it/s]

Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Kennen
Azir
Azir
Azir
Zilean
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir
Azir


In [12]:
freq_df["norm_support"].sum()

np.float64(1.0000000000000002)